# Análisis exploratorio — Importaciones de camiones (partida 8704229000)

Fuente: **Veritrade** — Perú Importaciones, partida `8704229000` (camiones diésel), Ene-2023 a Abr-2026.
Tabla estructurada generada por `scripts/extract_descripcion.py`.

> **Nota de calidad**: `modelo`~79% y `version`~29% por el formato "lista por comas" en ~21% de filas; `largo_mm` mezcla unidades (mm vs metros). Los campos clave (marca, vin, chasis, cilindrada, ejes, CIF) tienen 99–100% de cobertura.


In [ ]:
import os
# correr desde la raíz del repo aunque el notebook esté en notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True

SRC = "outputs/veritrade_PE_I_8704229000_estructurado.xlsx"
df = pd.read_excel(SRC, sheet_name="estructurado")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
# limpiar relleno de espacios en columnas de texto (campos de ancho fijo del origen)
obj = df.select_dtypes("object").columns
df[obj] = df[obj].apply(lambda s: s.str.strip())
print(df.shape)
df.head(3)


## 1. Vistazo general


In [ ]:
print("Filas:", len(df), "| Columnas:", df.shape[1])
print("Rango de fechas:", df["fecha"].min().date(), "→", df["fecha"].max().date())
df.dtypes


### Cobertura por columna (% no nulo)


In [ ]:
cob = (df.notna().mean() * 100).sort_values()
ax = cob.plot(kind="barh", color="#4C78A8")
ax.set_title("Cobertura por columna (%)"); ax.set_xlabel("% no nulo")
plt.tight_layout(); plt.show()
cob.round(0).to_frame("pct_no_nulo")


## 2. Calidad de datos — chequeos rápidos


In [ ]:
dups = df["vin"].dropna().duplicated().sum()
print("VINs duplicados:", dups)
en_metros = (df["largo_mm"] < 50).sum()
print("Filas con largo_mm en metros (sospechoso):", en_metros)
print("Sin modelo:", df["modelo"].isna().sum(), f"({df['modelo'].isna().mean():.0%})")
df[["cilindrada_cc","peso_bruto","potencia_hp","carga_util","largo_mm","anio_modelo"]].describe().round(1)


## 3. Evolución temporal de las importaciones


In [ ]:
por_mes = df.set_index("fecha").resample("MS").size()
ax = por_mes.plot(marker="o", color="#4C78A8")
ax.set_title("Unidades importadas por mes"); ax.set_ylabel("unidades"); ax.set_xlabel("")
plt.tight_layout(); plt.show()
df["fecha"].dt.year.value_counts().sort_index().to_frame("unidades")


## 4. Marcas — participación de mercado


In [ ]:
top_marcas = df["marca"].value_counts().head(15)
ax = top_marcas.iloc[::-1].plot(kind="barh", color="#54A24B")
ax.set_title("Top 15 marcas por unidades importadas"); ax.set_xlabel("unidades")
plt.tight_layout(); plt.show()
(df["marca"].value_counts(normalize=True).head(10) * 100).round(1).to_frame("market_share_%")


## 5. Importadores — panorama competitivo


In [ ]:
top_imp = df["importador"].value_counts().head(15)
ax = top_imp.iloc[::-1].plot(kind="barh", color="#E45756")
ax.set_title("Top 15 importadores por unidades"); ax.set_xlabel("unidades")
plt.tight_layout(); plt.show()
dw = df[df["importador"].str.contains("DW", case=False, na=False)]
print("Filas de importadores con 'DW':", len(dw))
dw["importador"].value_counts().head()


### ¿Qué marcas trae cada top-importador?


In [ ]:
top5 = df["importador"].value_counts().head(6).index
(df[df["importador"].isin(top5)]
   .groupby(["importador","marca"]).size()
   .sort_values(ascending=False).groupby(level=0).head(3)).to_frame("unidades")


## 6. Origen y logística


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
df["pais_origen"].value_counts().head(8).iloc[::-1].plot(kind="barh", ax=axes[0,0], title="País de origen")
df["puerto_embarque"].value_counts().head(8).iloc[::-1].plot(kind="barh", ax=axes[0,1], title="Puerto de embarque")
df["aduana"].value_counts().head(8).iloc[::-1].plot(kind="barh", ax=axes[1,0], title="Aduana")
df["estado"].value_counts().plot(kind="bar", ax=axes[1,1], title="Estado (nuevo/usado)", rot=0)
plt.tight_layout(); plt.show()


## 7. Valor comercial (CIF / FOB)


In [ ]:
print("CIF total importado (USD): {:,.0f}".format(df["cif_usd"].sum()))
print("FOB total (USD):          {:,.0f}".format(df["fob_usd"].sum()))
g = (df.groupby("marca")
       .agg(unidades=("cif_usd","size"), cif_total=("cif_usd","sum"), cif_prom_unit=("cif_usd","mean"))
       .sort_values("unidades", ascending=False).head(12))
g["cif_total"] = g["cif_total"].round(0); g["cif_prom_unit"] = g["cif_prom_unit"].round(0)
g


In [ ]:
ax = g.sort_values("cif_prom_unit")["cif_prom_unit"].plot(kind="barh", color="#B279A2")
ax.set_title("Valor CIF promedio por unidad — top marcas (USD)"); ax.set_xlabel("USD/unidad")
plt.tight_layout(); plt.show()


## 8. Especificaciones técnicas


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
df["cilindrada_cc"].clip(0, 16000).plot(kind="hist", bins=40, ax=axes[0,0], title="Cilindrada (cc)")
df["potencia_hp"].clip(0, 600).plot(kind="hist", bins=40, ax=axes[0,1], title="Potencia (HP)")
df["peso_bruto"].clip(0, 55000).plot(kind="hist", bins=40, ax=axes[1,0], title="Peso bruto")
df["ejes"].value_counts().sort_index().plot(kind="bar", ax=axes[1,1], title="Nº de ejes", rot=0)
plt.tight_layout(); plt.show()


## 9. Tabla dinámica: marca × año (volumen)


In [ ]:
pivot = (df.assign(anio=df["fecha"].dt.year)
           .pivot_table(index="marca", columns="anio", values="vin", aggfunc="count", fill_value=0))
pivot["total"] = pivot.sum(axis=1)
pivot.sort_values("total", ascending=False).head(15)


## Cierre y próximos pasos

- Tabla apta para EDA de mercado: volumen, marcas, importadores, origen, valor CIF y specs.
- **Limitaciones**: `modelo`/`version` incompletos (~21% formato sucio) y `largo_mm` con unidades mezcladas.
- Si `modelo`/`version` resultan clave, evaluar la *fase B* (LLM solo para ese ~21%).
